<div style="border-left:6px solid #ae0000; padding:6px 20px; margin-bottom:4px;">
<h1 style="margin:0; color:#0d2741;">Generadores de Números Aleatorios</h1>
<p style="margin:4px 0 0 0; color:#0d2741; font-size:1.12em;">NumPy Random, Semillas y Distribuciones</p>
<p style="margin:6px 0 0 0; color:#444; font-size:1.05em;"><em>Estadística Computacional para la Toma de Decisiones</em></p>
</div>

**Magíster en Ciencia de Datos e Inteligencia Artificial** · Universidad Andrés Bello  
Maidana, J.P. (2026)

---

> **Cómo usar este notebook.** Ejecuta las celdas en orden, comenzando por **«Preparación del entorno»**. Todo el código usa la **API nueva** de NumPy (`numpy.random.Generator`), recomendada para trabajo reproducible.

## Preparación del entorno

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import FancyBboxPatch, Circle   # solo para los diagramas/figuras

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams['figure.dpi'] = 100

print("numpy:", np.__version__)

<div style="background-color:#fffceb; border-left:5px solid #f9a825; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#f57f17; font-size:1.05em;">📝&nbsp; Nota — Sobre la reproducibilidad de los valores exactos</p>
<p style="margin:0;">La secuencia base de <code>Generator.random()</code> es <strong>estable entre versiones</strong> de NumPy: la demo de reproducibilidad de la Sección 3 produce exactamente los mismos números que el apunte. En cambio, los métodos de transformación (<code>uniform</code>, <code>normal</code>, <code>binomial</code>, …) pueden cambiar su algoritmo interno entre versiones, por lo que los <em>valores puntuales</em> de algunas simulaciones (p. ej. la estimación de π por tamaño) pueden diferir levemente de los del apunte. Lo que <strong>no</strong> cambia es el comportamiento estadístico: medias, varianzas y convergencias coinciden con lo esperado.</p>
</div>

## 1. Introducción: La Aleatoriedad en Computación

Consideremos una situación común en ciencia de datos. Necesitas dividir un dataset en entrenamiento y prueba. Ejecutas tu código hoy y obtienes un accuracy de 87%. Lo ejecutas mañana y obtienes 84%. ¿Cambió el modelo? ¿Los datos? ¿Tu código tiene un error?

La respuesta puede ser más simple: **no fijaste la semilla aleatoria**. Cada vez que ejecutas, la división es diferente y los resultados varían no porque haya un problema, sino porque la aleatoriedad no está controlada.

Desde simulaciones Monte Carlo hasta validación cruzada, desde la inicialización de redes neuronales hasta el bootstrapping, la aleatoriedad controlada es una herramienta esencial. Permite **simulación** (modelar procesos estocásticos), **muestreo** (seleccionar subconjuntos representativos), **validación** (crear conjuntos de entrenamiento/prueba), **bootstrapping** (estimar intervalos de confianza) y **generación sintética** (crear datos para pruebas).

Sin embargo, hay una paradoja: en computadoras, la aleatoriedad "verdadera" no existe de forma natural. Lo que usamos son **números pseudoaleatorios**: secuencias determinísticas que parecen aleatorias pero son reproducibles.

In [ ]:
# Diagrama: del valor inicial (semilla) a los números pseudoaleatorios
fig, ax = plt.subplots(figsize=(11, 3.4))
ax.set_xlim(0, 12); ax.set_ylim(0, 4); ax.axis('off')

def _box(x, titulo, sub, fc):
    ax.add_patch(FancyBboxPatch((x-1.5, 1.5), 3.0, 1.4,
        boxstyle="round,pad=0.1,rounding_size=0.15", fc=fc, ec='#333', lw=1.4))
    ax.text(x, 2.4, titulo, ha='center', fontsize=11, fontweight='bold')
    ax.text(x, 1.9, sub, ha='center', fontsize=8.5, color='#444')

_box(1.8,  "Semilla",  "(seed)",            '#cfd8e8')
_box(6.0,  "Generador", "Algoritmo PRNG",   '#c8e6c9')
_box(10.2, "Números",  "Pseudoaleatorios",  '#ffe0b2')

for x1, x2, lbl in [(3.3, 4.5, 'inicializa'), (7.5, 8.7, 'produce')]:
    ax.annotate('', xy=(x2, 2.2), xytext=(x1, 2.2),
        arrowprops=dict(arrowstyle='-|>', color='#0d2741', lw=1.8))
    ax.text((x1+x2)/2, 2.5, lbl, ha='center', fontsize=8, color='#0d2741')

for x, t in [(1.8, "Valor inicial\n(reproducible)"), (6.0, "PCG64\nMersenne Twister"),
             (10.2, "Secuencia\ndeterminística")]:
    ax.text(x, 0.9, t, ha='center', fontsize=8, color='#666')

plt.tight_layout()
plt.show()

<div style="background-color:#fffceb; border-left:5px solid #f9a825; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#f57f17; font-size:1.05em;">📝&nbsp; Nota — Lo que aprenderás</p>
<ol style="margin:0;">
<li>Cómo funcionan los generadores de números pseudoaleatorios (PRNG).</li>
<li>Por qué las semillas son fundamentales para la reproducibilidad.</li>
<li>La nueva API de NumPy (<code>numpy.random.Generator</code>).</li>
<li>Diferencias entre la API antigua y la nueva.</li>
<li>Cómo generar muestras desde distribuciones comunes.</li>
<li>Aplicaciones prácticas en análisis de datos.</li>
<li>Mejores prácticas para trabajo reproducible.</li>
</ol>
</div>

## 2. Fundamentos: Números Pseudoaleatorios

### 2.1 ¿Qué es un PRNG?

<div style="background-color:#e8f1fb; border-left:5px solid #1565c0; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#0d47a1; font-size:1.05em;">📘&nbsp; Definición — Generador de Números Pseudoaleatorios (PRNG)</p>
<p style="margin:0 0 6px 0;">Un <strong>PRNG</strong> es un algoritmo determinístico que produce secuencias de números con propiedades estadísticas similares a las de números aleatorios verdaderos.</p>
<p style="margin:0 0 4px 0;"><strong>Características clave:</strong></p>
<ul style="margin:0;">
<li><strong>Determinístico:</strong> misma semilla ⇒ misma secuencia.</li>
<li><strong>Reproducible:</strong> fundamental para debugging y validación científica.</li>
<li><strong>Estadísticamente aleatorio:</strong> pasa tests de aleatoriedad.</li>
<li><strong>Eficiente:</strong> rápido de computar.</li>
<li><strong>Periodo largo:</strong> no se repite en un tiempo razonable.</li>
</ul>
</div>

El determinismo de los PRNG es, paradójicamente, una *ventaja*: permite reproducibilidad científica (otros pueden verificar resultados), debugging exacto (reproducir bugs consistentemente), pruebas unitarias determinísticas y comparación justa de algoritmos en condiciones idénticas.

### 2.2 Algoritmos Principales

| Algoritmo | Época | Características |
|---|---|---|
| Mersenne Twister | 1997–2019 | Periodo muy largo ($2^{19937}-1$); usado en NumPy legacy |
| PCG64 | 2019–presente | Más rápido, mejor calidad estadística; *default* en NumPy moderno |
| Philox | Alternativa | Paralelizable; adecuado para GPUs |
| SFC64 | Alternativa | Muy rápido; periodo adecuado |

<div align="center"><em>Tabla 2.1 — Generadores disponibles en NumPy.</em></div>

<div style="background-color:#fff4e6; border-left:5px solid #e65100; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#bf360c; font-size:1.05em;">⭐&nbsp; Importante — NumPy tiene DOS APIs de random</p>
<p style="margin:0;">La <strong>API nueva</strong> (post-2019) es más flexible, más rápida y evita los problemas del estado global. Se recomienda para todo código nuevo. La comparación de sintaxis es:</p>
</div>

**API Legacy** (antigua, pre-2019) — usa un estado global:

```python
import numpy as np
np.random.seed(42)        # Estado global
x = np.random.randn(100)  # Usa el estado global
```

**API Nueva** (recomendada, post-2019) — usa un generador explícito:

```python
import numpy as np
rng = np.random.default_rng(42)  # Generador explícito
x   = rng.normal(size=100)        # Usa ese generador específico
```

## 3. Semillas: Control de la Aleatoriedad

### 3.1 ¿Qué es una Semilla?

<div style="background-color:#e8f1fb; border-left:5px solid #1565c0; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#0d47a1; font-size:1.05em;">📘&nbsp; Definición — Semilla (seed)</p>
<p style="margin:0 0 8px 0;">Una <strong>semilla</strong> es un valor entero que inicializa el estado interno del PRNG. Proporciona el punto de partida para la secuencia determinística:</p>
<div style="text-align:center; margin:6px 0;">Misma semilla + Mismo algoritmo = Misma secuencia de números</div>
<p style="margin:8px 0 0 0;">La semilla es típicamente un entero entre 0 y $2^{32}-1$ (o más grande, según el generador).</p>
</div>

Las semillas son necesarias por cinco razones: **reproducibilidad científica** (otros replican exactamente tus resultados), **debugging** (reproducir bugs consistentemente), **testing** (pruebas unitarias determinísticas), **comparación justa** (múltiples modelos en condiciones idénticas) y **validación** (auditorías que requieren reproducibilidad).

### 3.2 Cómo Usar Semillas en NumPy

El siguiente ejemplo demuestra la propiedad central: la misma semilla reproduce exactamente los mismos números.

In [ ]:
# Crear generador con semilla
rng = np.random.default_rng(42)
print("Primera ejecución:")
print(rng.random(5))

# Recrear con la misma semilla
rng = np.random.default_rng(42)
print("\nSegunda ejecución (misma semilla):")
print(rng.random(5))   # Números idénticos

# Diferente semilla
rng = np.random.default_rng(123)
print("\nCon semilla diferente:")
print(rng.random(5))   # Números diferentes

**Salida esperada** (estos valores son estables entre versiones de NumPy):

```text
Primera ejecución:
[0.77395605 0.43887844 0.85859792 0.69736803 0.09417735]

Segunda ejecución (misma semilla):
[0.77395605 0.43887844 0.85859792 0.69736803 0.09417735]

Con semilla diferente:
[0.68235186 0.05382102 0.22035987 0.18437181 0.1759059 ]
```

<div style="background-color:#fdecea; border-left:5px solid #c62828; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#b71c1c; font-size:1.05em;">⚠️&nbsp; Advertencia — Errores Comunes con Semillas</p>
<p style="margin:0 0 6px 0;"><strong>Error 1:</strong> fijar la semilla global cuando código intermedio puede usar <code>random</code> y modificar el estado compartido. <strong>Mejor práctica:</strong> usar un generador local (<code>rng</code>), que siempre es reproducible.</p>
<p style="margin:0;"><strong>Error 2:</strong> fijar la semilla <em>dentro</em> de un bucle, lo que reinicia la secuencia y genera el mismo número en cada iteración. <strong>Solución:</strong> crear el generador una sola vez, fuera del bucle.</p>
</div>

```python
# Error 1 — estado global (NO recomendado)
np.random.seed(42)
data1 = np.random.randn(100)
# ...código que puede modificar el estado global...
data2 = np.random.randn(100)   # puede no ser reproducible

# Mejor — generador local
rng   = np.random.default_rng(42)
data1 = rng.normal(size=100)
data2 = rng.normal(size=100)    # siempre reproducible
```

```python
# Error 2 — semilla dentro del bucle (INCORRECTO): siempre el mismo número
for i in range(10):
    np.random.seed(42)
    print(np.random.random())

# Correcto — generador creado una sola vez
rng = np.random.default_rng(42)
for i in range(10):
    print(rng.random())         # secuencia continua
```

**Demostración en vivo del Error 2** (el bucle incorrecto repite el mismo valor; el correcto avanza la secuencia):

In [ ]:
print("INCORRECTO (semilla dentro del bucle) -> mismo número 5 veces:")
for i in range(5):
    np.random.seed(42)
    print(f"  {np.random.random():.6f}")

print("\nCORRECTO (un generador, fuera del bucle) -> secuencia continua:")
rng = np.random.default_rng(42)
for i in range(5):
    print(f"  {rng.random():.6f}")

### 3.3 ¿Qué Semilla Usar?

- **Para producción/publicación:** semilla fija y documentada (ej. 42, 123, 2026).
- **Para exploración:** `None` (basada en el tiempo del sistema).
- **Para experimentos robustos:** ejecutar con múltiples semillas y reportar estadísticas agregadas.

```python
rng = np.random.default_rng(42)          # Fija y documentada
rng = np.random.default_rng()            # Sin semilla (time-based)
rng = np.random.default_rng(seed=2026)   # Control explícito
```

## 4. Distribuciones: Muestreo Estadístico

### 4.1 Distribuciones Disponibles

| Distribución | Método | Uso típico |
|---|---|---|
| Uniforme $[0,1)$ | `random()` | Muestreo general, simulación |
| Uniforme $[a,b)$ | `uniform()` | Rango específico |
| Normal | `normal()` | Errores, mediciones naturales |
| Normal estándar | `standard_normal()` | Z-scores, normalización |
| Binomial | `binomial()` | Éxitos en $n$ intentos |
| Poisson | `poisson()` | Conteos, eventos raros |
| Exponencial | `exponential()` | Tiempos entre eventos |
| Enteros | `integers()` | Muestreo discreto |
| Selección | `choice()` | Selección de elementos |
| Permutación | `permutation()` | Reordenar datos |

<div align="center"><em>Tabla 4.1 — Distribuciones comunes en NumPy.</em></div>

<div style="background-color:#e8f1fb; border-left:5px solid #1565c0; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#0d47a1; font-size:1.05em;">📘&nbsp; Definición — Distribución Uniforme Continua</p>
<p style="margin:0 0 6px 0;">Todos los valores en $[a, b)$ tienen la misma probabilidad:</p>
$$f(x) = \begin{cases} \dfrac{1}{b-a} & \text{si } a \leq x < b \\[4pt] 0 & \text{en otro caso} \end{cases}$$
<p style="margin:6px 0 0 0;"><strong>Media:</strong> $\mu = \dfrac{a+b}{2}$ &nbsp;&nbsp;&nbsp; <strong>Varianza:</strong> $\sigma^2 = \dfrac{(b-a)^2}{12}$</p>
</div>

In [ ]:
rng = np.random.default_rng(42)

u1 = rng.random(10_000)                          # Uniforme [0, 1)
u2 = rng.uniform(low=-5, high=5, size=10_000)    # Uniforme [-5, 5)

print(f"Media u1: {u1.mean():.4f}  (esperado: 0.5)")
print(f"Media u2: {u2.mean():.4f}  (esperado: 0.0)")
print(f"Var u2:   {u2.var():.4f}   (esperado: {100/12:.4f})")

<div style="background-color:#e8f1fb; border-left:5px solid #1565c0; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#0d47a1; font-size:1.05em;">📘&nbsp; Definición — Distribución Normal</p>
$$f(x) = \frac{1}{\sigma\sqrt{2\pi}}\,\exp\!\left(-\frac{1}{2}\left(\frac{x-\mu}{\sigma}\right)^{\!2}\right)$$
<p style="margin:6px 0 0 0;"><strong>Regla empírica:</strong> 68% de los datos en $[\mu\pm\sigma]$, 95% en $[\mu\pm2\sigma]$, 99.7% en $[\mu\pm3\sigma]$.</p>
</div>

In [ ]:
rng = np.random.default_rng(42)

z = rng.standard_normal(10_000)                  # N(0, 1)
x = rng.normal(loc=100, scale=15, size=10_000)   # N(100, 15^2)

print(f"Media: {x.mean():.2f}  (esperado: 100)")
print(f"Desv.Est: {x.std():.2f}  (esperado: 15)")

within_1s = np.abs(x - 100) <= 15
print(f"Dentro de 1 sigma: {within_1s.mean()*100:.1f} %  (esperado: 68 %)")

within_2s = np.abs(x - 100) <= 30
print(f"Dentro de 2 sigma: {within_2s.mean()*100:.1f} %  (esperado: 95 %)")

### 4.2 Otras Distribuciones Importantes

In [ ]:
rng = np.random.default_rng(42)

# Binomial: número de caras en 10 lanzamientos
binomial = rng.binomial(n=10, p=0.5, size=1_000)
print(f"Binomial - Media: {binomial.mean():.2f}  (esperado: n*p = 5)")
print(f"           Desv.: {binomial.std():.2f}   (esperado: sqrt(n*p*(1-p)) ~ 1.58)")

# Poisson: llamadas por hora (lambda = 3)
poisson = rng.poisson(lam=3.0, size=1_000)
print(f"\nPoisson  - Media: {poisson.mean():.2f}  (esperado: lambda = 3)")
print(f"           Var:   {poisson.var():.2f}   (esperado: lambda = 3)")

# Exponencial: tiempo entre llegadas (escala = 5 min)
exponen = rng.exponential(scale=5.0, size=1_000)
print(f"\nExponen. - Media: {exponen.mean():.2f}  (esperado: 5)")
print(f"           Desv.: {exponen.std():.2f}   (esperado: 5)")

**Visualización de las distribuciones.** Generamos muestras grandes de cada distribución y observamos su forma característica.

In [ ]:
rng = np.random.default_rng(42)
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
ax = axes.ravel()

ax[0].hist(rng.uniform(-5, 5, 10_000), bins=40, color='#90caf9', edgecolor='white')
ax[0].set_title('Uniforme [-5, 5)', fontweight='bold')

ax[1].hist(rng.normal(100, 15, 10_000), bins=40, color='#a5d6a7', edgecolor='white')
ax[1].set_title(r'Normal ($\mu$=100, $\sigma$=15)', fontweight='bold')

ax[2].hist(rng.standard_normal(10_000), bins=40, color='#80cbc4', edgecolor='white')
ax[2].set_title('Normal estándar N(0,1)', fontweight='bold')

vals, cnts = np.unique(rng.binomial(10, 0.5, 10_000), return_counts=True)
ax[3].bar(vals, cnts, color='#ffcc80', edgecolor='white')
ax[3].set_title('Binomial (n=10, p=0.5)', fontweight='bold')

vals, cnts = np.unique(rng.poisson(3.0, 10_000), return_counts=True)
ax[4].bar(vals, cnts, color='#ce93d8', edgecolor='white')
ax[4].set_title(r'Poisson ($\lambda$=3)', fontweight='bold')

ax[5].hist(rng.exponential(5.0, 10_000), bins=40, color='#ef9a9a', edgecolor='white')
ax[5].set_title('Exponencial (escala=5)', fontweight='bold')

for a in ax:
    a.set_ylabel('Frecuencia')
fig.suptitle('Distribuciones generadas con numpy.random.Generator', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Muestreo y Selección

### 5.1 Funciones de Muestreo

In [ ]:
rng  = np.random.default_rng(42)
data = np.array([10, 20, 30, 40, 50, 60, 70, 80, 90, 100])

# 1. Con reemplazo (puede haber duplicados)
print("Con reemplazo:", rng.choice(data, size=5, replace=True))

# 2. Sin reemplazo (sin duplicados)
print("Sin reemplazo:", rng.choice(data, size=5, replace=False))

# 3. Muestreo ponderado
probs = np.array([0.05, 0.05, 0.1, 0.1, 0.2,
                  0.2,  0.1,  0.1, 0.05, 0.05])
ws = rng.choice(data, size=1_000, p=probs)
print("\nMuestreo ponderado - frecuencias:")
vals, cnts = np.unique(ws, return_counts=True)
for v, c in zip(vals, cnts):
    print(f"  {v:>4}: {c:>4} veces  ({c/10:.1f} %)")

# 4. Permutación (nuevo orden aleatorio)
print("\nPermutación:", rng.permutation(data))

# 5. Shuffle in-place
data_copy = data.copy()
rng.shuffle(data_copy)
print("Shuffle:", data_copy)

### 5.2 Aplicación: Train/Test Split

Una implementación propia de la división entrenamiento/prueba que ilustra cómo `permutation` + una semilla producen particiones reproducibles.

In [ ]:
def train_test_split(X, y, test_size=0.2, random_state=None):
    """
    Dividir datos en entrenamiento y prueba.

    Parameters
    ----------
    X            : array (n_samples, n_features)
    y            : array (n_samples,)
    test_size    : float — proporción de test
    random_state : int   — semilla

    Returns
    -------
    X_train, X_test, y_train, y_test
    """
    rng      = np.random.default_rng(random_state)
    n        = len(X)
    n_test   = int(n * test_size)
    idx      = rng.permutation(n)
    test_idx = idx[:n_test]
    trn_idx  = idx[n_test:]
    return X[trn_idx], X[test_idx], y[trn_idx], y[test_idx]

# Ejemplo
X = np.arange(100).reshape(100, 1)
y = np.arange(100)

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Total: {len(X)}  Train: {len(Xtr)}  Test: {len(Xte)}")
print(f"Primeros 5 índices de test: {yte[:5]}")

# Verificar reproducibilidad
Xtr2, Xte2, ytr2, yte2 = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"¿Mismos índices? {np.array_equal(yte, yte2)}")

## 6. Aplicaciones Prácticas

### 6.1 Simulación Monte Carlo

Estimamos $\pi$ generando puntos uniformes en el cuadrado $[-1, 1]^2$ y midiendo la fracción que cae dentro del círculo unitario (cuya área es $\pi$, frente al área 4 del cuadrado).

In [ ]:
def estimate_pi(n_samples, random_state=None):
    """Estimar pi mediante Monte Carlo."""
    rng  = np.random.default_rng(random_state)
    x, y = rng.uniform(-1, 1, n_samples), rng.uniform(-1, 1, n_samples)
    return 4 * np.sum(x**2 + y**2 <= 1) / n_samples

print(f"pi verdadero: {np.pi:.6f}\n")
for n in [100, 1_000, 10_000, 100_000, 1_000_000]:
    est = estimate_pi(n, random_state=42)
    print(f"n={n:>9,}: pi ~ {est:.6f}   error = {abs(est-np.pi):.6f}")

<div style="background-color:#fffceb; border-left:5px solid #f9a825; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#f57f17; font-size:1.05em;">📝&nbsp; Nota — Sobre los valores de esta simulación</p>
<p style="margin:0;">Los valores puntuales por cada $n$ dependen de la versión de NumPy (el método <code>uniform</code> puede variar entre versiones), por lo que pueden diferir levemente de los del apunte. Lo importante es el patrón: <strong>el error tiende a 0 a medida que crece $n$</strong>, con una convergencia del orden de $1/\sqrt{n}$ (cuadruplicar $n$ reduce el error a la mitad, aproximadamente).</p>
</div>

**Visualización del método** (puntos dentro vs fuera del círculo unitario):

In [ ]:
rng = np.random.default_rng(42)
n = 3_000
x, y = rng.uniform(-1, 1, n), rng.uniform(-1, 1, n)
dentro = x**2 + y**2 <= 1
est = 4 * dentro.sum() / n

fig, ax = plt.subplots(figsize=(6.2, 6.2))
ax.scatter(x[dentro], y[dentro], s=6, color='#2e7d32', alpha=0.5, label='Dentro')
ax.scatter(x[~dentro], y[~dentro], s=6, color='#c62828', alpha=0.5, label='Fuera')
ax.add_patch(Circle((0, 0), 1, fill=False, color='#0d2741', lw=2))
ax.add_patch(plt.Rectangle((-1, -1), 2, 2, fill=False, color='#0d2741', lw=1.2))
ax.set_aspect('equal'); ax.set_xlim(-1.05, 1.05); ax.set_ylim(-1.05, 1.05)
ax.set_title(fr'Monte Carlo: $\pi \approx$ {est:.4f}  (n={n:,})', fontweight='bold')
ax.legend(loc='upper right', framealpha=0.9)
plt.tight_layout()
plt.show()

### 6.2 Bootstrapping

El bootstrap estima la incertidumbre de un estadístico remuestreando los datos *con reemplazo* muchas veces. El intervalo de confianza se obtiene de los percentiles de la distribución bootstrap.

In [ ]:
def bootstrap_ic(data, statistic=np.mean, n_boot=10_000,
                 confidence=0.95, random_state=None):
    """IC bootstrap percentil."""
    rng  = np.random.default_rng(random_state)
    n    = len(data)
    boot = np.array([
        statistic(rng.choice(data, size=n, replace=True))
        for _ in range(n_boot)
    ])
    alpha = 1 - confidence
    return (np.percentile(boot, 100 * alpha / 2),
            np.percentile(boot, 100 * (1 - alpha / 2)),
            boot)

rng  = np.random.default_rng(42)
data = rng.normal(loc=50, scale=10, size=100)
lo, hi, dist = bootstrap_ic(data, random_state=42)

print(f"Media observada:  {data.mean():.2f}")
print(f"IC 95 % bootstrap: [{lo:.2f}, {hi:.2f}]")
print(f"Desv. bootstrap:  {dist.std():.2f}")

**Visualización** de la distribución bootstrap (reutiliza `data` y `dist`):

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.hist(dist, bins=50, color='#90caf9', edgecolor='white')
ax.axvline(data.mean(), color='#0d2741', lw=2, label=f'Media obs. = {data.mean():.2f}')
ax.axvline(lo, color='#c62828', ls='--', lw=1.8, label=f'IC 95% = [{lo:.2f}, {hi:.2f}]')
ax.axvline(hi, color='#c62828', ls='--', lw=1.8)
ax.set_title('Distribución bootstrap de la media (10.000 remuestreos)', fontweight='bold')
ax.set_xlabel('Media remuestreada'); ax.set_ylabel('Frecuencia')
ax.legend()
plt.tight_layout()
plt.show()

### 6.3 Generación de Datos Sintéticos

Creamos un dataset de regresión con relación conocida ($y = 3x_1 + 2x_2 - 5 + \text{ruido}$) para validar modelos o hacer pruebas.

In [ ]:
def generate_synthetic(n=1_000, random_state=None):
    """
    Genera dataset sintético para regresión:
    y = 3*x1 + 2*x2 - 5 + ruido
    """
    rng   = np.random.default_rng(random_state)
    x1    = rng.normal(0, 1, n)
    x2    = rng.normal(0, 1, n)
    noise = rng.normal(0, 0.5, n)
    y     = 3 * x1 + 2 * x2 - 5 + noise
    return np.column_stack([x1, x2]), y

X, y = generate_synthetic(n=1_000, random_state=42)
print(f"Shape X: {X.shape}   Shape y: {y.shape}")
print(f"Corr x1-y: {np.corrcoef(X[:,0], y)[0,1]:.3f}")
print(f"Corr x2-y: {np.corrcoef(X[:,1], y)[0,1]:.3f}")

## 7. Mejores Prácticas

A continuación, una guía de patrones recomendados. Los primeros cuatro son plantillas de referencia; el quinto es un test ejecutable.

**1. Usar la API nueva**

```python
rng  = np.random.default_rng(42)   # recomendado
data = rng.normal(size=100)

# Evitar:
# np.random.seed(42); data = np.random.randn(100)
```

**2. Documentar la semilla**

```python
RANDOM_SEED = 42   # constante al inicio del script
rng = np.random.default_rng(RANDOM_SEED)
# "Todos los experimentos usaron semilla 42 para reproducibilidad."
```

**3. Pasar generadores como argumentos**

```python
def mi_funcion(data, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    return data + rng.normal(size=len(data))
```

**4. Múltiples semillas para robustez**

```python
results = [run_experiment(data, np.random.default_rng(s))
           for s in range(10)]
print(f"Accuracy: {np.mean(results):.3f} ± {np.std(results):.3f}")
```

**5. Verificar reproducibilidad con un test unitario** (ejecutable):

In [ ]:
def test_reproducibility():
    r1 = np.random.default_rng(42).random(100)
    r2 = np.random.default_rng(42).random(100)
    assert np.allclose(r1, r2), "No es reproducible"
    print("Reproducibilidad verificada")

test_reproducibility()

## 8. Comparación: API Legacy vs Nueva

| Operación | Legacy (antigua) | Nueva (Generator) |
|---|---|---|
| Inicializar | `np.random.seed(42)` | `rng = np.random.default_rng(42)` |
| Uniforme $[0,1)$ | `np.random.rand()` | `rng.random()` |
| Normal estándar | `np.random.randn()` | `rng.standard_normal()` |
| Normal | `np.random.normal()` | `rng.normal()` |
| Enteros | `np.random.randint()` | `rng.integers()` |
| Choice | `np.random.choice()` | `rng.choice()` |
| Shuffle | `np.random.shuffle()` | `rng.shuffle()` |

<div align="center"><em>Tabla 8.1 — Comparación de APIs.</em></div>

**Ventajas de la nueva API:** no usa estado global (más segura en código paralelo), es más rápida (especialmente PCG64), tiene mejor calidad estadística, permite múltiples generadores independientes y ofrece una interfaz más consistente.

## 9. Resumen y Puntos Clave

1. Los números pseudoaleatorios son **determinísticos pero estadísticamente aleatorios**. Esta propiedad es una ventaja para la reproducibilidad.
2. Las **semillas** controlan la secuencia de números generados: misma semilla ⇒ misma secuencia.
3. NumPy tiene **dos APIs**: legacy (antigua) y Generator (nueva). Se recomienda la nueva para todo código nuevo.
4. `np.random.default_rng(seed)` crea un **generador independiente**, más seguro que el estado global.
5. NumPy soporta **muchas distribuciones**: uniforme, normal, binomial, Poisson, exponencial, y más.
6. Funciones como `choice()`, `permutation()` y `shuffle()` facilitan el **muestreo y reordenamiento**.
7. Para reproducibilidad científica, **documenta siempre las semillas** usadas.
8. En experimentos robustos, ejecuta con **múltiples semillas diferentes** y reporta estadísticas agregadas.

### Cierre

<div style="background-color:#ffffff; border:1px solid #d9d9d9; border-left:60px solid #0d2741; border-radius:6px; padding:16px 20px; margin:6px 0;">
<span style="font-size:1.4em;">💡</span>&nbsp; La aleatoriedad controlada es fundamental en ciencia de datos. Usar semillas apropiadamente no solo permite reproducibilidad científica, sino que facilita el debugging, la validación de resultados y la comparación justa de algoritmos. La nueva API de NumPy (<code>Generator</code>) ofrece mejor rendimiento, mayor flexibilidad y más seguridad que la API legacy, y adoptarla desde el principio es una de las mejores inversiones en calidad de código computacional.
</div>